# Notebook 3 — GARCH Volatility Modelling
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Understand volatility and why GARCH is the right tool
2. Calculate historical (realised) volatility for BSE stocks
3. Test for ARCH effects (volatility clustering)
4. Fit a GARCH(1,1) model using the `arch` library
5. Evaluate the model (AIC, BIC, residual diagnostics)
6. Forecast future volatility
7. Use the `GarchModel` class from `src/model.py`
8. Save the fitted model to disk with `joblib`

## 1. Setup

In [ ]:
import sys
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from arch import arch_model
from arch.unitroot import ADF
import joblib

sys.path.insert(0, os.path.join('..', 'src'))
from data import SQLRepository, YFinanceAPI
from model import GarchModel

DB_PATH     = os.path.join('..', 'database', 'stock_data.db')
MODELS_DIR  = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

connection = sqlite3.connect(DB_PATH)
repo       = SQLRepository(connection=connection)

print('Setup complete.')
print(f'Models will be saved to: {os.path.abspath(MODELS_DIR)}')

## 2. What is Volatility?

Volatility measures how much an asset's price **fluctuates over time**.

- **Historical (realised) volatility** — standard deviation of past returns, annualised.
- **Implied volatility** — forward-looking, derived from options prices (not covered here).
- **Conditional volatility** — what GARCH models: volatility that varies *over time* based on recent shocks.

**Why GARCH?** Stock returns show *volatility clustering* — large moves tend to follow large moves, and calm periods cluster too. GARCH captures this by making today's variance depend on yesterday's shock and yesterday's variance.

## 3. Load Data from SQLite

In [ ]:
TICKER = 'RELIANCE.NS'

# If not in DB yet, fetch it first
if not repo.table_exists(TICKER):
    print(f'{TICKER} not in DB — fetching from Yahoo Finance...')
    api = YFinanceAPI()
    df_raw = api.get_daily_data(TICKER, '2015-01-01', '2024-12-31')
    repo.insert_table(TICKER, df_raw, if_exists='replace')

df = repo.read_table(TICKER)
df['returns'] = df['Close'].pct_change() * 100
df.dropna(inplace=True)

returns = df['returns']
print(f'Loaded {len(returns):,} return observations for {TICKER}')
print(f'Date range: {returns.index.min().date()}  →  {returns.index.max().date()}')
returns.describe()

## 4. Historical (Realised) Volatility

In [ ]:
# Annualised rolling volatility (30-day and 90-day windows)
df['vol_30d']  = returns.rolling(30).std()  * np.sqrt(252)
df['vol_90d']  = returns.rolling(90).std()  * np.sqrt(252)

overall_vol = returns.std() * np.sqrt(252)
print(f'Annualised volatility (full sample): {overall_vol:.2f}%')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(returns.index, returns, color='steelblue', linewidth=0.7, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_title(f'{TICKER} — Daily Returns (%)', fontsize=12)
axes[0].set_ylabel('Return (%)')

axes[1].plot(df.index, df['vol_30d'],  color='firebrick', linewidth=1.0, label='30-day rolling vol')
axes[1].plot(df.index, df['vol_90d'],  color='orange',    linewidth=1.2, label='90-day rolling vol', alpha=0.8)
axes[1].axhline(overall_vol, color='black', linewidth=0.8, linestyle='--', label=f'Full-sample vol = {overall_vol:.1f}%')
axes[1].set_title('Annualised Rolling Volatility', fontsize=12)
axes[1].set_ylabel('Volatility (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Test for ARCH Effects (Volatility Clustering)

The ARCH-LM test checks whether squared residuals are autocorrelated — i.e. whether volatility clusters.

In [ ]:
from arch.univariate import ARCH

# ADF test: check returns are stationary (required for GARCH)
adf_result = ADF(returns)
print('=== Augmented Dickey-Fuller Test ===')
print(f'ADF statistic : {adf_result.stat:.4f}')
print(f'p-value       : {adf_result.pvalue:.6f}')
print(f'Stationary    : {adf_result.pvalue < 0.05}  (reject unit root at 5%)')

print()

# ARCH-LM test
test_model = arch_model(returns, p=1, q=0)  # ARCH(1) as base
fit0 = test_model.fit(disp='off')
arch_test = fit0.arch_lm_test(lags=10)
print('=== ARCH-LM Test (lags=10) ===')
print(f'LM statistic : {arch_test.stat:.4f}')
print(f'p-value      : {arch_test.pvalue:.6f}')
print(f'ARCH effects : {arch_test.pvalue < 0.05}  (GARCH is justified)' if arch_test.pvalue < 0.05
      else f'ARCH effects : {arch_test.pvalue < 0.05}  (may not need GARCH)')

## 6. Fit the GARCH(1,1) Model

**GARCH(1,1)** model:

$$\sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2$$

- $\omega$ — long-run variance (constant)
- $\alpha_1$ — ARCH effect (sensitivity to recent shocks)
- $\beta_1$ — GARCH effect (persistence of volatility)
- $\alpha_1 + \beta_1 < 1$ for stationarity

In [ ]:
# Fit using arch library directly (for transparency)
garch_spec = arch_model(returns, p=1, q=1, rescale=False)
garch_fit = garch_spec.fit(disp='off')

print(garch_fit.summary())

In [ ]:
# Extract key parameters
params = garch_fit.params
omega  = params['omega']
alpha1 = params['alpha[1]']
beta1  = params['beta[1]']

print(f'omega   (ω)  = {omega:.6f}   — baseline variance')
print(f'alpha1  (α₁) = {alpha1:.4f}   — ARCH effect (shock sensitivity)')
print(f'beta1   (β₁) = {beta1:.4f}   — GARCH effect (volatility persistence)')
print(f'α₁ + β₁     = {alpha1 + beta1:.4f}   — persistence (< 1 = stationary)')
print()
print(f'AIC = {garch_fit.aic:.2f}')
print(f'BIC = {garch_fit.bic:.2f}')

## 7. Conditional Volatility Plot

In [ ]:
cond_vol = garch_fit.conditional_volatility  # daily std dev (%)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cond_vol.index, y=cond_vol,
    mode='lines', name='GARCH(1,1) Conditional Vol',
    line=dict(color='firebrick', width=1)
))
fig.update_layout(
    title=f'{TICKER} — GARCH(1,1) Conditional Volatility (Daily Std Dev %)',
    xaxis_title='Date', yaxis_title='Volatility (%)',
    template='plotly_white', height=420
)
fig.show()

## 8. Compare Model Orders — Find the Best (p, q)

In [ ]:
# Grid search over (p, q) combinations
results = []

for p in [1, 2]:
    for q in [1, 2]:
        try:
            spec = arch_model(returns, p=p, q=q, rescale=False)
            fit  = spec.fit(disp='off')
            results.append({'p': p, 'q': q, 'AIC': fit.aic, 'BIC': fit.bic})
        except Exception as e:
            results.append({'p': p, 'q': q, 'AIC': np.nan, 'BIC': np.nan})

results_df = pd.DataFrame(results).sort_values('AIC')
print('Model comparison (lower AIC/BIC = better):')
print(results_df.to_string(index=False))
best_p, best_q = results_df.iloc[0][['p', 'q']].astype(int).values
print(f'\nBest model: GARCH({best_p}, {best_q})')

## 9. Forecast Volatility

In [ ]:
HORIZON = 5  # trading days ahead

forecast = garch_fit.forecast(horizon=HORIZON, reindex=False)
variance_forecast = forecast.variance.iloc[-1]

# Annualise: daily variance → annual volatility
vol_forecast = np.sqrt(variance_forecast) * np.sqrt(252)

print(f'GARCH(1,1) volatility forecast for {TICKER} — next {HORIZON} trading days:')
print()
for i, (label, val) in enumerate(vol_forecast.items(), start=1):
    print(f'  Day {i} ({label}):  {val:.2f}% annualised volatility')

## 10. Using the `GarchModel` Class from `src/model.py`

Everything above wrapped in a clean, reusable class.

In [ ]:
# Build model using the class
gm = GarchModel(ticker=TICKER, repo=repo)
gm.wrangle_data(n_observations=2000)
gm.fit(p=1, q=1)

print(f'AIC : {gm.aic:.4f}')
print(f'BIC : {gm.bic:.4f}')

vol = gm.predict_volatility(horizon=5)
print('\n5-day annualised volatility forecast:')
for label, value in vol.items():
    print(f'  {label} : {value:.2f}%')

## 11. Save the Model to Disk

In [ ]:
# Use the built-in helper to generate a datestamped path
model_path = GarchModel.build_model_path(TICKER, models_dir=MODELS_DIR)
saved_path = gm.dump(model_path)

print(f'Model saved to: {saved_path}')

# Verify we can reload it
gm_reload = GarchModel(ticker=TICKER)
gm_reload.load(saved_path)

vol_reload = gm_reload.predict_volatility(horizon=5)
print('\nReloaded model — 5-day forecast:')
for label, value in vol_reload.items():
    print(f'  {label} : {value:.2f}%')

## 12. Residual Diagnostics

In [ ]:
std_resid = garch_fit.std_resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(std_resid.index, std_resid, color='steelblue', linewidth=0.6, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Standardised Residuals')
axes[0].set_ylabel('Std Residual')

axes[1].hist(std_resid, bins=80, density=True, color='steelblue', alpha=0.6)
from scipy import stats
x = np.linspace(std_resid.min(), std_resid.max(), 200)
axes[1].plot(x, stats.norm.pdf(x, 0, 1), 'r-', linewidth=2, label='N(0,1)')
axes[1].set_title('Distribution of Standardised Residuals')
axes[1].set_xlabel('Std Residual')
axes[1].legend()

plt.tight_layout()
plt.show()

# Ljung-Box test on squared residuals — should show no autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(std_resid**2, lags=[10], return_df=True)
print(f'Ljung-Box p-value (squared residuals, lag=10): {lb["lb_pvalue"].values[0]:.4f}')
print('(p > 0.05 means GARCH has adequately modelled volatility clustering)')

In [ ]:
connection.close()
print('Done. Models are saved in:', os.path.abspath(MODELS_DIR))

## Summary

- BSE returns exhibit strong **volatility clustering** confirmed by the ARCH-LM test.
- **GARCH(1,1)** is typically the best model by AIC/BIC for Indian equity returns.
- The `GarchModel` class in `src/model.py` wraps fit → forecast → save/load cleanly.
- Saved models live in `models/<ticker>_<date>.pkl`.

**Next:** Notebook 4 — deploying the model via a FastAPI server.